# "Surplus Exergy" 
Need to think of a more accurate name later

## Libraries used

In [73]:
import numpy as np
import pandas as pd

# =============================================
# Core Scientific/LCA (Life Cycle Assessment) Libraries
# Required for Brightway2.5 (LCA framework)
# Always place these at the top, as they are the main dependencies for your project
# =============================================
import os
import bw2io as bi # Brightway2 library for data import/export
import bw2data as bd # Core Brightway2 library for LCA database management

# =============================================
# Data Processing and Numerical Libraries
# Used for data manipulation, math, and array operations
# =============================================
import numpy as np  # Numerical computing (arrays, math operations), only version for np.nan # numpy==1.26.4
import pandas as pd # Data manipulation and analysis (DataFrames)
import math         # Basic math functions
import csv          # CSV file reading/writing
import os           # Operating system interfaces (file paths, environment variables)
import re           # Regular expressions (text pattern matching)

# =============================================
# File System and Path Handling
# Used for file and directory operations
# =============================================
from pathlib import Path  # Object-oriented filesystem paths

# =============================================
# Time and Delay Utilities
# Used for introducing delays or measuring time
# =============================================
from time import sleep  # Introduce delays (e.g., for web scraping)
import time             # Time-related functions (e.g., `time.time()`)

# =============================================
# Chemistry and Data Retrieval
# Used for chemical data and external API access
# =============================================
import pubchempy as pcp                 # Interface to PubChem (chemical data)
from pubchempy import PubChemHTTPError  # Unique CF assignment for each flow that contains copper
import requests                         # HTTP requests (e.g., API calls)
from mendeleev import element           # Periodic table data (element properties)

15:00:49+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


## Functions

In [74]:
# Functions for TheRy

# 1. The name of each flow will be cleaned using re
def clean_chemical_name(name):
    # This detects some of the organic molecules (e.g. Ethane, 1,1-difluoro-, HFC-152a)
    pattern1 = r"^(.+?),\s+(.+?)-(?:,\s+.*|$)"
    # This pattern simple detects the flows with a comma.
    pattern2 = r"(?<=[a-zA-Z]),.*"

    # First, check if the flow satisfies pattern1
    match = re.search(pattern1,name)
    # If there is a match, then we change its format (e.g. from Ethane, 1,1-difluoro-, HFC-152a to 1,1-difluoroEthane)
    if match:
        name = re.sub(pattern1, r"\g<2>\g<1>", name)
    # If not, we apply the 2nd pattern
    else:
        name = re.sub(pattern2, "", name)
    
    # Second, remove Roman numerals at the end (e.g., Aluminium III -> Aluminium) if applicable
    name = re.sub(r"\s+[IVXLCDM]+$", "", name)
    
    # 3. Third the word 'ion' or 'ions' at the end (e.g., Copper ion -> Copper) if applicable
    # Using re.IGNORECASE makes it catch 'Ion', 'ION', or 'ion'
    name = re.sub(r"\s+ions?$", "", name, flags=re.IGNORECASE)
    
    return name.strip()


# 2. Then, it will be fed to the API to find the chemical. If it is found, the original flow name, the chemical IUPAC name, molecular composition, and the molecular weight will be made into a dictionary
def chem_comp(mf):
    # First of all, we need to use re to cut the molecular formula into pieces
    # Regex breakdown:
    # ([A-Z][a-z]?) -> Matches an Uppercase letter potentially followed by a lowercase (the element)
    # (\d*)         -> Matches zero or more digits following the element (the count)
    pattern = r"([A-Z][a-z]?)(\d*)"
    
    matches = re.findall(pattern, mf)
    
    parts = []

    for elem,count in matches:
        # If the count is empty (like in 'Cl'), it means there is 1 atom
        count = int(count) if count else 1
        parts.append([elem,count])
    
    return parts

def has_elem(parts): 
    for elem, count in parts:
        if elem not in CF1_dict.keys(): #if it is not included in the list of elements
            print(f"{elem} is not included in the scope.")
            continue
        else:
            return True
    return None

# 3. Calculates for the CF of the given molecular formular
def cf_calculator(mf,mw,parts):
    cf=0 # place holder for the 

    for elem, count in parts:
        if elem in CF1_dict.keys():
            # Find the atomic mass of the element first
            el_data = element(elem) # from mendeley library
            mass = float(el_data.atomic_weight) # from mendeley library
            # Compute for the contribution of this element in the final mf
            cf += count*(mass/mw)*CF1_dict[elem]
        else:
            continue

    return round(cf,3)

## Import data for CF1

## $CF_1$ Calculation

$CF_1 = \frac{\partial g}{\partial CMT} = -\frac{A \beta e^{\alpha}}{CMT^2}(\frac{A}{CMT}-1)^{\beta - 1}$

### Data Structure recommendation
Organize the $\alpha$, $\beta$, $R_{URR}$, and CMT data of each metal in a dictionary,  <br>
Just like how the TheRy was corresponded to each metal. <br>

### CF assignment
Search for the element name in the chemical formula of the flow, adjust the CF based on the mass percent <br>

### Import data and compute for CF1 computation

In [75]:
# Import the data from local excel file
OGD_df = pd.read_excel("Ore-GradeDeclineConstants.xlsx")

# Grouping the numerator and denominator explicitly
numerator = OGD_df['URR'] * OGD_df['beta'] * np.exp(OGD_df['alpha'])
denominator = OGD_df['CME']**2.0 #Forced into a float computation because the number is too large to handle

# Explicitly separated calculation
OGD_df['CF1'] = -(numerator / denominator) * ((OGD_df['URR'] / OGD_df['CME']) - 1)**(OGD_df['beta'] - 1)
OGD_df = OGD_df[OGD_df['Metal']!= "Gold"]
#OGD_df

In [76]:
# Build a data dictionary with the element symbols as the keys
CF1_dict = dict(zip(OGD_df.iloc[:,1],OGD_df.iloc[:,6]))
#CF1_dict

### Unique CF assignment and adding the method data to the database

In [77]:
# Set up the project and database
project_name = "Ecoinvent3.4"
bd.projects.set_current(project_name)

# Retrieves credentials that are stored in the os.
username = os.getenv("EI_USERNAME")
password = os.getenv("EI_PASSWORD")

# Use this code if the project is not imported yet
#bi.import_ecoinvent_release("3.4", "cutoff", username, password)

# 1. Initialize the database object
biosphere = bd.Database('ecoinvent-3.4-biosphere')

bd.databases

Databases dictionary with 2 object(s):
	ecoinvent-3.4-biosphere
	ecoinvent-3.4-cutoff

In [78]:
# we need a if-else filter here to see if the flow is elementary. If yes, do not go through the cf_calculator

method_data = []
extra_info = []

not_found = []
not_emission = [] # can be used to check for the flows that aren't considered

i=0
for flow in biosphere:
    cf = None
    parts = None #reset parts every time
    # Only consider the elementary flows going into the biosphere
    # Find flows that contain any of the search terms
    if isinstance(flow.get('categories'), tuple) and len(flow['categories']) > 0 and flow['unit'].lower() == 'kilogram':

        flow_name_cleaned = clean_chemical_name(flow['name'])

        sleep(0.3) #Slow down the request to the API to avoid sudden shut down of connection

        max_retries = 3
        for attempt in range(max_retries):
            try:
                compound = pcp.get_compounds(flow_name_cleaned,'name')
                if compound: 
                    c = compound[0]
                    mf = c.molecular_formula
                    mw = c.molecular_weight
            
                    parts = chem_comp(mf)

                    # If the flow contains only one element, we can directly use its CF value
                    if len(parts)==1:
                        if has_elem(parts): #if the only element is a part of the metal group
                            cf = CF1_dict.get(parts[0][0])   
                            print(f"The cf of {mf} is decided to be {cf}")
                        else:
                            print(f"{flow['name']} does not contain any element of the scope.")       
                    # If the flow contains a compound, use the cf_calculator
                    elif len(parts)>1:
                        if has_elem(parts):
                            cf = cf_calculator(mf,mw,parts)
                            print(f"The cf of {mf} is calculated to be {cf}")
                        else:
                            print(f"{flow['name']} does not contain any element of the scope.")   
                    else: 
                        print(f"The molecular compound does not contain any element.")
                    
                    # Combine the original flow name, 
                    if cf is not None:
                        method_data.append((flow.key,float(cf)))
                        extra_info.append([flow.key,flow['name'],flow_name_cleaned,mf,mw,float(cf)])
                    else: 
                        extra_info.append([flow.key,flow['name'],flow_name_cleaned,mf,mw,None])
                        print('This compound does not contain any element of the scope.')
                else:
                    print(f"Compound not found for: {flow['name']}")
                    not_found.append([flow.key,flow['name']])

            except PubChemHTTPError as e:
                # Check if it's a 502 or another server error
                if "502" in str(e) and attempt < max_retries - 1:
                    print(f"Server hit a 502 for {flow_name_cleaned}. Retrying in 2 seconds... (Attempt {attempt + 1}/{max_retries})")
                    time.sleep(2)  # Give the server a moment to recover
                else:
                    print(f"Failed to fetch {flow_name_cleaned} after multiple attempts or met a different error: {e}")
                    # Handle or log the permanent failure here
                    break
    else:
        not_emission.append([flow.key,flow['name']])


C is not included in the scope.
H is not included in the scope.
Na is not included in the scope.
O is not included in the scope.
Sodium formate does not contain any element of the scope.
This compound does not contain any element of the scope.
C is not included in the scope.
H is not included in the scope.
Na is not included in the scope.
O is not included in the scope.
Sodium formate does not contain any element of the scope.
This compound does not contain any element of the scope.
Failed to fetch Sodium formate after multiple attempts or met a different error: PubChem HTTP Error 502 Bad Gateway
C is not included in the scope.
H is not included in the scope.
O is not included in the scope.
Acetone does not contain any element of the scope.
This compound does not contain any element of the scope.
C is not included in the scope.
H is not included in the scope.
O is not included in the scope.
Acetone does not contain any element of the scope.
This compound does not contain any element of

In [79]:
PC_df = pd.DataFrame(method_data)
PC_df.to_csv("data/PubChem_Method_EI34.csv", index=False)

In [80]:
PC_df = pd.DataFrame(extra_info)
NF_df = pd.DataFrame(not_found)
NInc_df = pd.DataFrame(not_emission)
PC_df.to_csv("data/PubChem_Info_EI34.csv", index=False)
NF_df.to_csv("data/PubChem_NotFound_EI34.csv", index=False)
NInc_df.to_csv("data/PubChem_NotEmission_EI34.csv", index=False)

### Encode the method data to the project file in BrightWay

In [81]:
# 1. Define the method name as a tuple for the desired hierarchy
method_name_tuple = (
    "Exergoecology Methods from the Department of Environment Science of Radboud University",
    "Ore-Grade Decline Potential",
    "Applied to 17 elements included in the SOG method of Vieira et al."  # Updated name
)

# 2. Define metadata for the method, including the unit and a description
method_metadata = {
    'unit': 'change in ore grade per kg of metal extracted',
    'description': 'This LCIA method is one out of the 3 steps of the currently developing surplus exergy method.'
                    'It models the decrease of ore-grade with the progression of the extraction activities.'
                    'The impact score (IS) isn\'t the final value of the intended impact yet. '
                    'This IS needs to be fed to two more calculations to find out the exergy lost due to the dissipation in the product system.',
    'source': 'The values are taken from the appendix of ReCiPe 2016: https://www.rivm.nl/bibliotheek/rapporten/2016-0104.pdf',
    'version': '1.0',
    'num_cfs': len(method_data),
    'application': 'Output product-system metals characterization'
}

# 3. Create or load the Brightway2 Method object
method_object = bd.Method(method_name_tuple)

# 4. Forcefully register/write the method
try:
    # This will overwrite existing method
    method_object.register(**method_metadata) 
    method_object.write(method_data)
    
    print(f"\n✅ Successfully {'overwrote' if method_name_tuple in bd.methods else 'created'} method: {method_name_tuple}")
    print(f"   - Unit: {method_metadata['unit']}")
    print(f"   - Number of CFs: {len(method_data)}")
    
except Exception as e:
    print(f"\n❌ Failed to write method: {str(e)}")
    raise


# --- Verification ---
print("\n--- Verification ---")
if method_name_tuple in bd.methods:
    retrieved_method = bd.Method(method_name_tuple)
    loaded_data = retrieved_method.load()
    
    print(f"🔍 Method verification:")
    print(f"   Name: {retrieved_method.name}")
    print(f"   Metadata version: {retrieved_method.metadata.get('version', 'N/A')}")
    print(f"   Number of CFs loaded: {len(loaded_data)}")
    
    # Check for potential data loss
    if len(loaded_data) != len(method_data):
        print(f"⚠️  Warning: CF count mismatch. Expected {len(method_data)}, got {len(loaded_data)}")
    else:
        print("✅ CF count matches expected value")
        
    # Show sample CFs
    print("\nSample characterization factors (first 3):")
    for cf in loaded_data[:3]:
        flow = bd.get_activity(cf[0])
        print(f"   - {flow['name']}: {cf[1]} MJ-Eq")
else:
    print(f"❌ Error: Method {method_name_tuple} not found after writing attempt")

print("\nProcess completed")


✅ Successfully overwrote method: ('Exergoecology Methods from the Department of Environment Science of Radboud University', 'Ore-Grade Decline Potential', 'Applied to 17 elements included in the SOG method of Vieira et al.')
   - Unit: change in ore grade per kg of metal extracted
   - Number of CFs: 1140

--- Verification ---
🔍 Method verification:
   Name: ('Exergoecology Methods from the Department of Environment Science of Radboud University', 'Ore-Grade Decline Potential', 'Applied to 17 elements included in the SOG method of Vieira et al.')
   Metadata version: 1.0
   Number of CFs loaded: 1140
✅ CF count matches expected value

Sample characterization factors (first 3):
   - Glufosinate ammonium: -0.0 MJ-Eq
   - Glufosinate ammonium: -0.0 MJ-Eq
   - Glufosinate ammonium: -0.0 MJ-Eq

Process completed


## Step 2 and 3 calculation 

In [44]:
# initial concentration based on ore-grade formula
xi = np.exp(alpha)*(R_URR/CMT-1)**beta

# assuming the extraction of 1 ton copper 
dg = CF_Cu * 1000

print(f"The change in concentration is {dg}. The initial concentration is {xi}. The concentration after the extraction activity is {xi-dg}")

The change in concentration is -1.2312433333350088e-11. The initial concentration is 0.03705451064441092. The concentration after the extraction activity is 0.037054510656723355


#### Change in ore-grade in terms of exergy
R = 8.314 kJ/kmol•K <br>
Tº = 290.15K = 17ºC (hmmm)

$b_c(x)=-RTº[ln(x)+(1-x)ln(1-x)]$ <br>
$CF_2 = |{b_c(x_i-\Delta g)-b_c{x_i}}|$

In [ ]:
# Recreating the result from the book of Alicia
import math

R = 8.314
T = 290.15

xc = 0.0000664
xm = 0.01674
bc_xc = -R*T*(math.log(xc)+(1-xc)/xc*math.log(1-xc))
bc_xm = -R*T*(math.log(xm)+(1-xm)/xm*math.log(1-xm))

D_bc = bc_xc-bc_xm
D_bc_corr = D_bc/(63.55+55.85+32.07*2)

print(f"The concentration exergy is {bc_xc-bc_xm} kJ/kmol. And {D_bc_corr/1000} GJ/ton of ore.")

The concentration exergy is 13359.943359873785 kJ/kmol. And 0.07279036373473784 GJ/ton of ore.


In [25]:
# Trying the new calculation using the numbers from Vieira et al.
bc_xi = -R*T*(math.log(xi)+(1-xi)/xi*math.log(1-xi))
bc_xf = -R*T*(math.log(xi-dg)+(1-xi+dg)/(xi-dg)*math.log(1-xi+dg))

D_bc_Cu = bc_xi-bc_xf
print(f"The corresponding change in exergy is {D_bc_Cu/63.55} MJ/ton ore.")
print(f"The final impact score is {D_bc_Cu/63.55} MJ for this product system.")


The corresponding change in exergy is 1.285262706298982e-08 MJ/ton ore.
The final impact score is 1.285262706298982e-08 MJ for this product system.
